# Queueing theory - M/E2/n queueing table using Dask

This notebook produces the typical 'average waiting time in units of service time' table ($W_q / E[ST]$) for an M/E2/n queue, for 1 to 10 berths (servers) $n$ and berth occupancies (server utilisations) $\rho$ of 10 % to 90 %, as presented in for example the PIANC (2014) guideline. Every combination is simulated with 1,000,000 customers to approach steady-state values.

Every combination is simulated as `nr_replications` independent simulations (Dask tasks) of `nr_arr` customers, so `nr_arr * nr_replications` customers in total per table cell:
* one simulation of 100,000 customers takes about 3 s and about 350 MB of memory, so many workers can run side by side (a single simulation of 1 million customers would need several GB per worker);
* 90 cells x 10 replications = 900 tasks that Dask spreads evenly over all workers;
* the spread between the replications gives the statistical uncertainty of the table values.

Each simulation starts with an empty system; with 100,000 customers per simulation the effect of this start-up period on the averages is negligible. At high occupancies successive waiting times are strongly correlated, so these cells converge slowest.

The total computation time is roughly (number of cells) x (customers per cell) x 27 µs / (number of workers): for 90 cells with 1 million customers each about 40 minutes of CPU time, i.e. a few minutes on 12 local workers.

The calculations are distributed with [Dask](https://docs.dask.org/en/stable/deploying.html):
* locally, a `LocalCluster` starts several worker *processes* on your computer (processes rather than threads, because the simulation is pure Python and threads would be blocked by Python's global interpreter lock);
* to upscale, the same code can connect to a larger cluster (e.g. a HPC or cloud cluster) by replacing the `LocalCluster` (see section 2).

NB: Dask is an optional dependency. Install it in your environment with `pip install "dask[distributed,diagnostics]"` (or `pip install -e ".[dask]"` from the OpenQTSim folder).

In [2]:
# OpenQTSim parallel engine
import os
import openqtsim
from openqtsim.mt_engine import create_tasks, run_tasks, waiting_time_table, mmc_waiting_time_factor
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dask.distributed import Client, LocalCluster

print("This notebook is executed with OpenQTSim version {}".format(openqtsim.__version__))

This notebook is executed with OpenQTSim version v0.5.1


#### 0. Define input

In [ ]:
A = "M"                 # distribution of interarrival times (M: exponential, E2: erlang 2, ...)
S = "E2"               # distribution of service times (M: exponential, E2: erlang 2, ...)

servers = range(1, 11)  # numbers of berths/servers n to consider
utilisations = np.round(np.arange(0.1, 1.0, 0.1), 2)  # berth occupancy rho = lambda / (n * mu_server)
mu = 10                 # total service capacity of the system (customers per hour); each server serves mu / n

customers_per_cell = 1_000_000  # customers per combination of n and rho

nr_arr = 100_000        # customers per simulation (task)
nr_replications = customers_per_cell // nr_arr  # independent simulations per table cell
seed = 2024             # makes the whole table reproducible
n_workers = os.cpu_count()  # number of local Dask worker processes

#### 1. Create task list
Each task is one simulation: a combination of berth occupancy, number of berths and replication, with its own independent arrival and service seeds.

In [4]:
tasks = create_tasks(A, S, servers, utilisations, nr_arr, mu=mu, nr_replications=nr_replications, seed=seed)

print(f"{len(tasks):,} simulations of {nr_arr:,} customers each ({nr_arr * nr_replications:,} customers per table cell)")
pd.DataFrame(tasks).head()

900 simulations of 100,000 customers each (1,000,000 customers per table cell)


,A,S,c,nr_arr,lam,mu,replication,arrival_seed,service_seed
0,M,E2,1,100000,1.0,10,0,2855298535,925030728
1,M,E2,1,100000,1.0,10,1,1146676384,1811149567
2,M,E2,1,100000,1.0,10,2,524768821,2577401382
3,M,E2,1,100000,1.0,10,3,4067285479,2609318523
4,M,E2,1,100000,1.0,10,4,2651666590,174624609


#### 2. Start a Dask cluster
The client shows a link to the Dask dashboard (open it in your browser), where you can follow the workers and the progress of the tasks while they run.

To upscale beyond your own computer, replace the `LocalCluster` by a larger cluster; the rest of the notebook stays the same. For example on a SLURM cluster such as DelftBlue (with `pip install dask-jobqueue`):

```python
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(cores=8, processes=8, memory="16GB", walltime="01:00:00")
cluster.scale(jobs=10)  # 10 jobs x 8 worker processes = 80 workers
```

or connect to a running scheduler with `client = Client("tcp://scheduler-address:8786")`.

In [5]:
cluster = LocalCluster(n_workers=n_workers, threads_per_worker=1, processes=True)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 12
Total threads: 12,Total memory: 16.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:58531,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:58562,Total threads: 1
Dashboard: http://127.0.0.1:58573/status,Memory: 1.33 GiB
Nanny: tcp://127.0.0.1:58534,


#### 3. Run the simulations in parallel

In [6]:
%%time
results = run_tasks(tasks, client)

CPU times: user 33.6 s, sys: 12.9 s, total: 46.5 s
Wall time: 7min 44s


#### 4. Waiting time table
Average waiting time in units of average service time ($W_q / E[ST]$) for the M/E2/n queue, averaged over the replications:

In [7]:
def pianc_layout(df):
    """Rows: berth occupancy in %, columns: number of berths n."""
    df = df.copy()
    df.index = [f"{u:.0%}" for u in df.index]
    df.index.name = "Berth occup. (ρ/n)"
    df.columns.name = "Number of berths n"
    return df

table = waiting_time_table(tasks, results)
pianc_layout(table).style.format("{:.4f}")

Number of berths n,1,2,3,4,5,6,7,8,9,10
Berth occup. (ρ/n),,,,,,,,,,
10%,0.0830,0.0080,0.0011,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
20%,0.1875,0.0326,0.0081,0.0025,0.0008,0.0003,0.0001,0.0000,0.0000,0.0000
30%,0.3222,0.0764,0.0268,0.0108,0.0047,0.0023,0.0011,0.0005,0.0003,0.0002
40%,0.5026,0.1471,0.0611,0.0308,0.0159,0.0090,0.0054,0.0031,0.0019,0.0013
50%,0.7536,0.2534,0.1218,0.0679,0.0415,0.0267,0.0182,0.0119,0.0085,0.0061
60%,1.1213,0.4297,0.2297,0.1398,0.0914,0.0649,0.0463,0.0351,0.0261,0.0202
70%,1.7525,0.7273,0.4267,0.2712,0.1985,0.1449,0.1113,0.0873,0.0712,0.0578
80%,2.9903,1.3544,0.8172,0.5715,0.4137,0.3281,0.2669,0.2200,0.1848,0.1574
90%,6.6945,3.1904,2.0773,1.5617,1.1397,0.9044,0.7802,0.6690,0.5797,0.5023


Standard error of the table values (standard deviation between the replications divided by $\sqrt{\text{nr\_replications}}$):

In [8]:
std_error = waiting_time_table(tasks, results, aggfunc="std") / np.sqrt(nr_replications)
pianc_layout(std_error).style.format("{:.4f}")

Number of berths n,1,2,3,4,5,6,7,8,9,10
Berth occup. (ρ/n),,,,,,,,,,
10%,0.0003,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
20%,0.0009,0.0002,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
30%,0.0012,0.0004,0.0002,0.0001,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000
40%,0.0020,0.0009,0.0004,0.0003,0.0001,0.0001,0.0001,0.0001,0.0000,0.0000
50%,0.0021,0.0009,0.0009,0.0007,0.0005,0.0002,0.0002,0.0002,0.0002,0.0002
60%,0.0050,0.0022,0.0013,0.0013,0.0004,0.0006,0.0006,0.0006,0.0004,0.0002
70%,0.0143,0.0041,0.0050,0.0030,0.0023,0.0021,0.0009,0.0011,0.0004,0.0008
80%,0.0331,0.0136,0.0102,0.0068,0.0043,0.0058,0.0029,0.0026,0.0044,0.0023
90%,0.0910,0.0660,0.0570,0.0242,0.0225,0.0181,0.0134,0.0164,0.0130,0.0072


#### 5. Compare with the reference table
There is no simple closed-form solution for the M/E2/n queue (only for n = 1, the Pollaczek-Khinchine formula gives $W_q / E[ST] = 0.75\,\rho / (1-\rho)$). The simulated table is compared with the table of Groenveld (2001) for 1-8 berths and 30-90 % berth occupancy (given with 2 decimals).

The differences are shown as absolute values (simulated - reference), because relative differences are meaningless for the very small waiting times at low occupancies and many berths.

In [9]:
# M/E2/n reference table (source: Groenveld, 2001)
reference_name = "Groenveld (2001), M/E2/n"
reference = pd.DataFrame(
    [
        [0.32, 0.08, 0.03, 0.01, 0.00, 0.00, 0.00, 0.00],
        [0.50, 0.15, 0.06, 0.03, 0.02, 0.01, 0.01, 0.00],
        [0.75, 0.26, 0.12, 0.07, 0.04, 0.03, 0.02, 0.01],
        [1.13, 0.43, 0.23, 0.14, 0.09, 0.06, 0.05, 0.03],
        [1.75, 0.73, 0.42, 0.27, 0.19, 0.14, 0.11, 0.09],
        [3.00, 1.34, 0.82, 0.57, 0.42, 0.33, 0.27, 0.22],
        [6.75, 3.14, 2.01, 1.45, 1.12, 0.91, 0.76, 0.65],
    ],
    index=[0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9], columns=range(1, 9),
)

difference = (table - reference).reindex(index=table.index, columns=table.columns)
display(pianc_layout(reference).style.format("{:.4f}", na_rep="").set_caption(reference_name))
display(pianc_layout(difference).style.format("{:+.4f}", na_rep="").set_caption(f"Simulated - {reference_name}"))

Number of berths n,1,2,3,4,5,6,7,8
Berth occup. (ρ/n),,,,,,,,
30%,0.3200,0.0800,0.0300,0.0100,0.0000,0.0000,0.0000,0.0000
40%,0.5000,0.1500,0.0600,0.0300,0.0200,0.0100,0.0100,0.0000
50%,0.7500,0.2600,0.1200,0.0700,0.0400,0.0300,0.0200,0.0100
60%,1.1300,0.4300,0.2300,0.1400,0.0900,0.0600,0.0500,0.0300
70%,1.7500,0.7300,0.4200,0.2700,0.1900,0.1400,0.1100,0.0900
80%,3.0000,1.3400,0.8200,0.5700,0.4200,0.3300,0.2700,0.2200
90%,6.7500,3.1400,2.0100,1.4500,1.1200,0.9100,0.7600,0.6500


Number of berths n,1,2,3,4,5,6,7,8,9,10
Berth occup. (ρ/n),,,,,,,,,,
10%,,,,,,,,,,
20%,,,,,,,,,,
30%,+0.0022,-0.0036,-0.0032,+0.0008,+0.0047,+0.0023,+0.0011,+0.0005,,
40%,+0.0026,-0.0029,+0.0011,+0.0008,-0.0041,-0.0010,-0.0046,+0.0031,,
50%,+0.0036,-0.0066,+0.0018,-0.0021,+0.0015,-0.0033,-0.0018,+0.0019,,
60%,-0.0087,-0.0003,-0.0003,-0.0002,+0.0014,+0.0049,-0.0037,+0.0051,,
70%,+0.0025,-0.0027,+0.0067,+0.0012,+0.0085,+0.0049,+0.0013,-0.0027,,
80%,-0.0097,+0.0144,-0.0028,+0.0015,-0.0063,-0.0019,-0.0031,+0.0000,,
90%,-0.0555,+0.0504,+0.0673,+0.1117,+0.0197,-0.0056,+0.0202,+0.0190,,


In [10]:
fig = go.Figure()
colors = px.colors.qualitative.Plotly
for i, n in enumerate(table.columns):
    color = colors[i % len(colors)]
    fig.add_trace(go.Scatter(
        x=100 * table.index, y=table[n], error_y=dict(type="data", array=std_error[n]),
        mode="markers", marker=dict(color=color, size=8), name=f"n = {n} simulated", legendgroup=str(n),
    ))
    if n in reference.columns:
        fig.add_trace(go.Scatter(
            x=100 * reference.index, y=reference[n], mode="lines", line=dict(color=color),
            name=f"n = {n} {reference_name}", legendgroup=str(n),
        ))
fig.update_layout(
    title=f"{A}/{S}/n: average waiting time in units of service time ({nr_arr * nr_replications:,} customers per cell)",
    xaxis_title="berth occupancy ρ/n [%]", yaxis_title="W_q / E[ST] [-]", yaxis_type="log",
    height=600, template="plotly_white",
)
fig.show()

#### 6. Close the Dask cluster

In [11]:
client.close()
cluster.close()